# Criptografía de Curvas Elípticas: De la Teoría a Bitcoin

**Material fuente:** *Artículo de investigación sobre Criptografía de Curvas Elípticas* (695.744 Ingeniería Inversa y Análisis de Vulnerabilidades)
**Relacionado:** [El Truco de Wright (2016)](./A1-wright-trick.ipynb) | [Firma Sin Sentido (2018)](./A2-nonsense-signature.ipynb) | [Cuaderno del Artículo Original](./A3-original-paper.ipynb)

---

## Empezar por el Final: ¿Qué Problema Resuelve la ECC?

En Bitcoin, la ECC resuelve dos problemas:

**Problema 1 — el alias.** El libro contable es público. Todos pueden ver
cada transacción. Necesitas que la gente pueda enviarte dinero, pero no puedes
entregar tu clave privada — cualquiera que la vea podría gastar tus fondos.
Así que necesitas una **clave pública**: un alias derivado de tu clave privada
al que cualquiera puede enviar, pero que no revela nada sobre la clave en sí.
Esa es la función unidireccional: $P = d \times G$ — fácil de calcular,
imposible de revertir.

En Bitcoin esto es literal. Cuando alguien te envía bitcoin, la transacción
bloquea los fondos con un script que dice: *"solo alguien que pueda producir una
clave pública cuyo hash sea este valor, Y una firma válida de esa clave, puede
gastar esto."* Eso es **P2PKH** (Pay-to-Public-Key-Hash) — el alias ni siquiera
es la clave pública directamente, es un hash de ella. Solo revelas $P$ cuando
gastas, junto con la firma. El estándar actual, **Taproot** (P2TR), bloquea
directamente a la clave pública y usa firmas Schnorr en lugar de ECDSA — pero
la estructura de los dos problemas es la misma.

**Problema 2 — la prueba.** Cuando quieres gastar, necesitas demostrar que
controlas el alias — que conoces la clave privada detrás de él — sin revelarla.
No puedes mostrar una contraseña (el libro contable es público, todos la verían).
No puedes depender de un tercero de confianza (no existe). Necesitas una prueba
que solo tú puedas producir, que cualquiera pueda verificar, que no revele nada
sobre tu secreto y que no pueda ser repetida. **Esa es una firma digital.**

La ECC nos da ambas cosas: la función unidireccional para el alias, y la
estructura algebraica para la firma. Antes de la ECC, ya teníamos respuestas —
RSA (1977) y ElGamal (1985) ambos funcionan. El Módulo 4 cubre ElGamal en
detalle porque la ECC es literalmente el mismo esquema trasplantado a un grupo
más difícil. Pero esos esquemas anteriores necesitan claves enormes — 3072 bits
para seguridad de 128 bits. Eso es lento, derrochador e impráctico para
dispositivos limitados.

En 1985, Neal Koblitz y Victor Miller preguntaron independientemente: **¿qué pasaría
si ejecutáramos el mismo truco del logaritmo discreto, pero en una estructura
matemática diferente — una donde el problema es fundamentalmente más difícil?**

La estructura que encontraron fue **curvas elípticas sobre cuerpos finitos.** Misma
seguridad, una fracción del tamaño de clave:

| Seguridad | RSA / ElGamal | ECC | Ahorro |
|-----------|--------------|-----|--------|
| 80 bits   | 1024 bits    | 160 bits | 6x más pequeña |
| 128 bits  | 3072 bits    | 256 bits | 12x más pequeña |
| 256 bits  | 15360 bits   | 512 bits | 30x más pequeña |

### Lo que necesitamos de las matemáticas

Para resolver ambos problemas, necesitamos:

**Para el alias (Problema 1):**

- Una **función unidireccional** — fácil de calcular hacia adelante, imposible de revertir
- Respuesta ECC: $P = d \times G$ (multiplicación escalar en una curva)
- Hacia adelante: milisegundos. Revertir: más energía de la que contiene el sistema solar.

**Para la firma (Problema 2):**

- Una **operación de firma** — solo el poseedor del secreto puede producirla
  - ECDSA: $s = k^{-1}(z + r \cdot d) \bmod N$
  - Requiere la clave privada $d$. No hay otra forma de producir un $(r, s)$ válido.
  - Usa la función unidireccional *otra vez*: un $k$ aleatorio fresco $\to R = k \times G$ se convierte en parte de la firma.
- Una **verificación pública** — cualquiera puede comprobarla sin conocer el secreto
  - ECDSA: verificar si $R'_x = r$ donde $R' = (z/s)G + (r/s)P$
  - Usa solo la clave pública $P$, el hash del mensaje $z$, y la firma $(r, s)$.

### Pero ¿POR QUÉ funciona esto?

Funciona porque las curvas elípticas sobre cuerpos finitos nos dan una función
unidireccional donde "multiplicación" significa algo geométricamente elegante —
trazar líneas a través de puntos de la curva, encontrar intersecciones,
reflejar. Y porque los puntos de la curva forman un **grupo algebraico**, todas
las reglas familiares de la aritmética se cumplen, pero revertir la operación
(el **logaritmo discreto**) es computacionalmente inviable.

Para construir esta función unidireccional, necesitamos un entorno matemático
muy específico — uno donde los números se envuelven, donde las líneas paralelas
se encuentran en el infinito, y donde los puntos en una curva obedecen los
axiomas de grupo.

**Ese es el recorrido de estos cuadernos:** partiendo de lo que necesitamos
lograr, construimos la maquinaria matemática pieza por pieza, y la vemos
cobrar vida en código funcional.

---

## Cómo Leer Esto

Cada módulo responde una pregunta que el módulo anterior planteó:

```
"Necesitamos un esquema de firma digital con claves pequeñas"
  └─→ "Necesitamos una función unidireccional en una curva"
        └─→ Módulo 4: De ElGamal a ECC — ¿de dónde vino este esquema?
              └─→ Módulo 3: Operaciones de Puntos — ¿cómo "sumamos" y "multiplicamos"?
                    └─→ Módulo 2: Curvas Elípticas — ¿qué son estas curvas?
                          └─→ Módulo 1: Fundamentos Algebraicos — ¿qué reglas gobiernan este mundo?

Luego construimos de vuelta con la respuesta:
  Módulo 5: ECDSA — el esquema de firmas que Bitcoin usó de 2009 a 2021
  Módulo 6: Aplicaciones en Bitcoin — Schnorr, diseño de transacciones, enrutamiento cebolla
  Módulo 7: Ejercicios — pon a prueba tu comprensión
```

Cada módulo es un cuaderno separado (01 a 07). Trabájalos en orden, o salta
directamente al Módulo 3 (el código) y vuelve a los Módulos 1-2 (la teoría)
cuando quieras entender POR QUÉ funcionan las fórmulas.